# 实践 5.3  DeepBSA与QTG-Miner：从分析流程到可计算练习

《统计遗传与智能育种学》第五章 · 电子资源 `05c_bsa_advanced.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

DeepBSA方法包括仿真数据生成、QTL定位和多算法比较；QTG-Miner流程涉及selectTBNsd.pl、矩阵转置以及sd.pl等Perl/Shell调用。本实践（电子资源：05c_bsa_advanced.ipynb）构造三个以上混池的虚拟等位基因频率，比较ΔSNP-index、ED⁴与示范性multi_pool_score，并生成小型基因网络复现候选基因到已知阳性基因的最短距离概念。必须注意，multi_pool_score只是用于说明多池信号汇总思想的可运行统计量，并不是DeepBSA论文中的DL或K值算法，也不能复现其训练模型或阈值。

QTG-Miner流程：selectTBNsd.pl → 矩阵转换/Inf处理 → sd.pl候选基因打分  
DeepBSA原研究流程：仿真/真实多混池频率 → 连续SNP窗口特征 → DL或K算法/QTL信号。  
运行DeepBSA前应核对所选平台的维护状态，固定软件、模型及依赖版本；跨平台复现还需核对输入处理和默认参数。

## 输入、完整代码、实际输出与解释

输入与可复现条件：三个以上虚拟混池、预设QTL区间和候选基因证据；SEED=503；无需外部文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=503
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 503


In [2]:
n=650; pos=np.arange(n)*15000; qtl=4_800_000; base=rng.uniform(.15,.85,n); amp=.22*np.exp(-((pos-qtl)/700000)**2); pools=np.vstack([np.clip(base-amp,0.02,.98),np.clip(base-.35*amp,0.02,.98),np.clip(base+.35*amp,0.02,.98),np.clip(base+amp,0.02,.98)]); depths=rng.poisson(50,(4,n))+1; obs=rng.binomial(depths,pools)/depths; delta=obs[-1]-obs[0]; ed4=(np.sqrt(2.0)*np.abs(delta))**4
score=np.std(obs,axis=0)*np.abs(obs[-1]-obs[0]); rank=pd.DataFrame({'pos':pos,'delta':delta,'ED4':ed4,'multi_pool_score':score}).nlargest(12,'multi_pool_score'); print(rank.round(5).to_string(index=False))

# 小型无向基因网络：广度优先搜索（breadth-first search, BFS）求候选到已知阳性基因的最短距离
from collections import deque
genes=[f'G{i}' for i in range(16)]; edges=[('G0','G1'),('G1','G2'),('G2','G3'),('G2','G7'),('G7','G8'),('G8','G9'),('G4','G5'),('G5','G7'),('G10','G11'),('G11','G12'),('G12','G8'),('G13','G14')]; adj={g:set() for g in genes}
for a,b in edges: adj[a].add(b); adj[b].add(a)
def dist(src,targets):
 q=deque([(src,0)]); seen={src}
 while q:
  u,d=q.popleft()
  if u in targets:return d
  for v in adj[u]-seen: seen.add(v); q.append((v,d+1))
 return np.inf
positive={'G3','G9'}; cand=['G0','G4','G10','G13']; print(pd.DataFrame({'candidate':cand,'distance_to_positive':[dist(g,positive) for g in cand]}).to_string(index=False))


    pos   delta     ED4  multi_pool_score
4935000 0.54579 0.35494           0.11902
5070000 0.56190 0.39873           0.11290
4605000 0.55769 0.38694           0.11205
4680000 0.52121 0.29520           0.10828
4665000 0.54626 0.35616           0.10713
4950000 0.52361 0.30067           0.09720
4770000 0.47719 0.20741           0.09557
4725000 0.45937 0.17812           0.09368
4830000 0.47730 0.20759           0.08468
5085000 0.42662 0.13251           0.08288
4875000 0.46305 0.18389           0.08194
4860000 0.46187 0.18202           0.08170
candidate  distance_to_positive
       G0                   3.0
       G4                   4.0
      G10                   4.0
      G13                   inf


结果解释：ΔSNP-index、ED⁴与multi_pool_score对同一频率模式强调的量不同，因此峰排序并不完全一致；候选表中的distance_to_positive只是一个证据维度。本练习用于解释DeepBSA/QTG-Miner类问题的统计结构，不复现正式软件全部算法，也不把教学multi_pool_score称为DeepBSA模型。

可修改练习：增加一个噪声pool或降低其中一个pool的深度，比较三类统计量的峰排序；再补充并定义候选基因证据的合成评分，检验指标方向与权重对排序的影响。